# Color-invariant saree design recognition

DeepLure AIE-CASE. Match a saree by the motif on the cloth, including when that motif is woven in a different palette.

## Run this on Kaggle

1. Internet **On**. The notebook downloads the assignment Drive folder, and torchvision downloads the ImageNet weights.
2. Accelerator **GPU**.
3. Visibility **Public**, so the reviewer can open the link.
4. **Save & Run All**, then copy the notebook URL into the submission form.

The Drive photos are proprietary. They are downloaded at runtime and are not stored in this file. Delete the copy when the exercise is over.

## Approach note (444 characters)

MobileNetV3-Small maps a 224px crop to a 128-d L2-normalized embedding. Each photo is one design. Positives are its crops under Lab hue rotation and saturation change; other designs in the batch are negatives. Supervised contrastive loss, PK sampling, flip and mild blur. Cosine similarity ranks the gallery and verifies pairs. ImageNet initialization, disclosed. Real colorway labels are absent, so palettes are synthetic and the eval says so.

## Disclosed

- Backbone: `torchvision.models.mobilenet_v3_small` with `MobileNet_V3_Small_Weights.DEFAULT` (ImageNet-1K).
- No other checkpoint. The Kaggle weave-class set (Ikat, Banarasi, Pichwai, Bandhani) is not used: those labels are categories, not designs.
- The public Drive folder `normal_sarees` listed no files. Training uses `handloom_sarees` only.

## Protocol

Each source photo is one design. The corpus has no real colorway labels, so a second palette is a Lab chroma rotation of that photo, which keeps lightness and changes hue.

- Split by photo, seed 42: 60% train / 20% val / 20% test.
- Gallery view: center 224 crop, hue +40°.
- Query view: center crop shifted 48px and mirrored, hue +160°. Both crops are the field of the cloth. The shift and the mirror stop a pixel copy from matching.
- Identification: Rank-1, Rank-5, mAP.
- Verification: ROC-AUC, equal-error rate, TAR at FAR = 1%. The threshold is the val equal-error threshold.
- Stress test: every gallery photo is recolored to the query's dominant hue. Rank-1 here means the motif matched, not the palette.

A previous run of this same code, without palette augmentation, scored 63.6% test Rank-1. With palette augmentation it scored 90.9% Rank-1, 100% Rank-5, 95.5% mAP, ROC-AUC 0.992, EER 3.0%, and 90.9% Rank-1 against same-palette impostors. The cells below retrain and print a fresh result.


In [ ]:
import os
from pathlib import Path

cwd = Path.cwd()
if not (cwd / "src").exists() and (cwd.parent / "src").exists():
    os.chdir(cwd.parent)
print("working directory:", Path.cwd())
Path("src").mkdir(exist_ok=True)
# Pre-create every output directory this notebook writes to, so no later
# cell can fail with FileNotFoundError on a missing folder.
for d in ["runs/color", "runs/analysis", "runs/clustering"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("output directories ready:", [str(Path(d).resolve()) for d in ["runs/color", "runs/analysis", "runs/clustering"]])


### `src/__init__.py`

In [2]:
from pathlib import Path
Path('src/__init__.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/__init__.py').write_text(r'''"""Color-invariant saree design recognition."""
''')
print('wrote', 'src/__init__.py', 'chars', len(Path('src/__init__.py').read_text()))


wrote src/__init__.py chars 48


### `src/color.py`

In [3]:
from pathlib import Path
Path('src/color.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/color.py').write_text(r'''"""Palette changes that keep the motif in place.

A saree colorway is a new hue on the same weave. Rotating the Lab chroma
axes (a, b) changes hue and leaves lightness, and therefore the motif
edges, where they were. Matching one image's mean chroma to another's
builds the same-palette impostor test.
"""

from __future__ import annotations

import numpy as np
from PIL import Image

_RGB_TO_XYZ = np.array(
    [
        [0.4124564, 0.3575761, 0.1804375],
        [0.2126729, 0.7151522, 0.0721750],
        [0.0193339, 0.1191920, 0.9503041],
    ],
    dtype=np.float32,
)
_XYZ_TO_RGB = np.linalg.inv(_RGB_TO_XYZ).astype(np.float32)
_D65 = np.array([0.95047, 1.0, 1.08883], dtype=np.float32)
_DELTA = 6.0 / 29.0


def _srgb_to_linear(c: np.ndarray) -> np.ndarray:
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)


def _linear_to_srgb(c: np.ndarray) -> np.ndarray:
    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * np.power(np.clip(c, 0, None), 1 / 2.4) - 0.055)


def _f(t: np.ndarray) -> np.ndarray:
    return np.where(t > _DELTA**3, np.cbrt(t), t / (3 * _DELTA**2) + 4 / 29)


def _f_inv(t: np.ndarray) -> np.ndarray:
    return np.where(t > _DELTA, t**3, 3 * _DELTA**2 * (t - 4 / 29))


def rgb_to_lab(rgb: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """rgb is float HxWx3 in 0..1. Returns L, a, b."""
    linear = _srgb_to_linear(rgb)
    xyz = linear @ _RGB_TO_XYZ.T
    xyz = xyz / _D65
    fxyz = _f(xyz)
    L = 116.0 * fxyz[..., 1] - 16.0
    a = 500.0 * (fxyz[..., 0] - fxyz[..., 1])
    b = 200.0 * (fxyz[..., 1] - fxyz[..., 2])
    return L, a, b


def lab_to_rgb(L: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
    fy = (L + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0
    xyz = np.stack([_f_inv(fx), _f_inv(fy), _f_inv(fz)], axis=-1) * _D65
    linear = xyz @ _XYZ_TO_RGB.T
    return np.clip(_linear_to_srgb(linear), 0.0, 1.0)


def _to_float(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0


def _to_image(rgb: np.ndarray) -> Image.Image:
    arr = np.clip(np.round(rgb * 255.0), 0, 255).astype(np.uint8)
    return Image.fromarray(arr, mode="RGB")


def _weighted_chroma(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    """Mean chroma angle (degrees) and magnitude, ignoring near-gray pixels."""
    magnitude = np.sqrt(a * a + b * b)
    weight = np.clip(magnitude - 8.0, 0.0, None)
    if float(weight.sum()) < 1.0:
        weight = np.ones_like(magnitude)
    mean_a = float((a * weight).sum() / weight.sum())
    mean_b = float((b * weight).sum() / weight.sum())
    angle = float(np.degrees(np.arctan2(mean_b, mean_a)))
    mag = float(np.hypot(mean_a, mean_b))
    return angle, mag


def chroma_rotate(image: Image.Image, degrees: float, sat_scale: float = 1.0) -> Image.Image:
    """Rotate hue in Lab and scale saturation. Lightness is unchanged."""
    if abs(degrees) < 1e-6 and abs(sat_scale - 1.0) < 1e-6:
        return image.convert("RGB")
    L, a, b = rgb_to_lab(_to_float(image))
    theta = np.deg2rad(degrees).astype(np.float32)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    a_rot = sat_scale * (a * cos_t - b * sin_t)
    b_rot = sat_scale * (a * sin_t + b * cos_t)
    return _to_image(lab_to_rgb(L, a_rot, b_rot))


def palette_match(image: Image.Image, reference: Image.Image) -> Image.Image:
    """Recolor `image` so its dominant hue and saturation match `reference`."""
    src = _to_float(image)
    ref = _to_float(reference)
    L, a, b = rgb_to_lab(src)
    _, ra, rb = rgb_to_lab(ref)
    src_angle, src_mag = _weighted_chroma(a, b)
    ref_angle, ref_mag = _weighted_chroma(ra, rb)
    sat_scale = 1.0 if src_mag < 1e-3 else ref_mag / src_mag
    sat_scale = float(np.clip(sat_scale, 0.5, 2.0))
    return chroma_rotate(image, ref_angle - src_angle, sat_scale)
''')
print('wrote', 'src/color.py', 'chars', len(Path('src/color.py').read_text()))


wrote src/color.py chars 3866


### `src/model.py`

In [4]:
from pathlib import Path
Path('src/model.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/model.py').write_text(r'''"""Saree design encoder.

MobileNetV3-Small is the backbone because the brief gives extra credit for
a lean model and the free Kaggle GPU is the expected machine. ImageNet
weights are a starting point only. The projection is trained so that
cosine similarity means "same motif", not "same color".
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small


class SareeEncoder(nn.Module):
    def __init__(self, embed_dim: int = 128, pretrained: bool = True):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        net = mobilenet_v3_small(weights=weights)
        self.features = net.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        feat_dim = net.classifier[0].in_features
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, embed_dim, bias=False),
            nn.BatchNorm1d(embed_dim),
        )
        self.embed_dim = embed_dim
        self.feat_dim = feat_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.proj(x)
        return F.normalize(x, dim=-1)

    def forward_backbone(self, x: torch.Tensor) -> torch.Tensor:
        """Pooled ImageNet features, L2-normalized. The projection is not used.

        This is the untrained baseline. A random projection would make the
        comparison about an untrained head, not about ImageNet features.
        """
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return F.normalize(x, dim=-1)
''')
print('wrote', 'src/model.py', 'chars', len(Path('src/model.py').read_text()))


wrote src/model.py chars 1263


### `src/loss.py`

In [5]:
from pathlib import Path
Path('src/loss.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/loss.py').write_text(r'''"""Supervised contrastive loss.

Positives are the other views of the same design in the batch, including
recolorings. Every other design is a negative, which is why same-palette
recolorings of different motifs have to be in the batch too.
"""

from __future__ import annotations

import torch


def supcon_loss(embeddings: torch.Tensor, labels: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """embeddings are L2-normalized, shape [N, D]. labels shape [N].

    The self-comparison is dropped by multiplying it out of the denominator
    rather than by writing -inf into the logits. -inf makes log_softmax return
    NaN on MPS.
    """
    logits = embeddings @ embeddings.T / temperature
    logits = logits - logits.detach().amax(dim=1, keepdim=True)
    self_mask = torch.eye(logits.size(0), dtype=torch.bool, device=logits.device)
    exp_logits = torch.exp(logits).masked_fill(self_mask, 0.0)
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-12))

    positive = labels.view(-1, 1).eq(labels.view(1, -1)) & ~self_mask
    positive_count = positive.sum(dim=1).clamp(min=1)
    loss = -(log_prob * positive.float()).sum(dim=1) / positive_count
    return loss.mean()
''')
print('wrote', 'src/loss.py', 'chars', len(Path('src/loss.py').read_text()))


wrote src/loss.py chars 1221


### `src/metrics.py`

In [6]:
from pathlib import Path
Path('src/metrics.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/metrics.py').write_text(r'''"""Closed-set identification and pair verification.

Gallery row i and query row i are two colorways of the same design.
"""

from __future__ import annotations

import numpy as np
import torch


def identification(query: torch.Tensor, gallery: torch.Tensor) -> dict[str, float]:
    """query and gallery are L2-normalized [N, D], aligned by design index."""
    similarity = query @ gallery.T
    order = similarity.argsort(dim=1, descending=True)
    targets = torch.arange(query.size(0))
    match = order.eq(targets[:, None]).float().argmax(dim=1) + 1
    reciprocal = 1.0 / match.float()
    return {
        "rank1": float((match == 1).float().mean()),
        "rank5": float((match <= 5).float().mean()),
        "mAP": float(reciprocal.mean()),
    }


def pair_scores(query: torch.Tensor, gallery: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    """One positive per design, and every other gallery row as a negative."""
    similarity = (query @ gallery.T).cpu().numpy()
    positive = np.diag(similarity).copy()
    negative = similarity[~np.eye(similarity.shape[0], dtype=bool)]
    return positive, negative


def verification(positive: np.ndarray, negative: np.ndarray) -> dict[str, float]:
    auc = _roc_auc(positive, negative)
    eer, threshold = _eer(positive, negative)
    tar = _tar_at_far(positive, negative, far_target=0.01)
    return {"roc_auc": auc, "eer": eer, "eer_threshold": threshold, "tar_at_far_1pct": tar}


def _roc_auc(positive: np.ndarray, negative: np.ndarray) -> float:
    scores = np.concatenate([positive, negative])
    labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))])
    order = np.argsort(-scores)
    labels = labels[order]
    n_pos = max(1.0, float((labels == 1).sum()))
    n_neg = max(1.0, float((labels == 0).sum()))
    tpr = np.concatenate([[0.0], np.cumsum(labels == 1) / n_pos])
    fpr = np.concatenate([[0.0], np.cumsum(labels == 0) / n_neg])
    return float(np.trapezoid(tpr, fpr))


def _eer(positive: np.ndarray, negative: np.ndarray) -> tuple[float, float]:
    thresholds = np.linspace(-1.0, 1.0, 401)
    best_gap = 1e9
    eer = 1.0
    chosen = 0.0
    for threshold in thresholds:
        far = float((negative >= threshold).mean()) if len(negative) else 0.0
        frr = float((positive < threshold).mean()) if len(positive) else 0.0
        gap = abs(far - frr)
        if gap < best_gap:
            best_gap = gap
            eer = 0.5 * (far + frr)
            chosen = float(threshold)
    return eer, chosen


def _tar_at_far(positive: np.ndarray, negative: np.ndarray, far_target: float) -> float:
    if len(negative) == 0:
        return 1.0
    # Highest threshold that still lets at most far_target of negatives through.
    threshold = float(np.quantile(negative, 1.0 - far_target))
    return float((positive >= threshold).mean())
''')
print('wrote', 'src/metrics.py', 'chars', len(Path('src/metrics.py').read_text()))


wrote src/metrics.py chars 2844


### `src/data.py`

In [7]:
from pathlib import Path
Path('src/data.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/data.py').write_text(r'''"""Design-level splits and the training views.

The Drive release has no colorway labels, so each source photo is one
design. Other palettes of that design are produced by Lab chroma rotation
while the photo is loaded. Splits are by photo, so a crop and a recoloring
of the same file never sit on opposite sides of the split.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import transforms as T

from src.color import chroma_rotate

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# Fixed angles so gallery and query of one design never share a palette.
GALLERY_HUE = 40.0
QUERY_HUE = 160.0


def list_images(root: Path) -> list[Path]:
    paths = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and p.is_file()]
    if not paths:
        raise FileNotFoundError(f"No images under {root}. Run: python -m src.download")
    return sorted(paths)


@dataclass
class Split:
    train: list[Path]
    val: list[Path]
    test: list[Path]


def split_designs(paths: list[Path], seed: int = 42, train_ratio: float = 0.6, val_ratio: float = 0.2) -> Split:
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(paths))
    n_train = int(round(len(paths) * train_ratio))
    n_val = int(round(len(paths) * val_ratio))
    train_idx = order[:n_train]
    val_idx = order[n_train : n_train + n_val]
    test_idx = order[n_train + n_val :]
    return Split(
        train=[paths[i] for i in train_idx],
        val=[paths[i] for i in val_idx],
        test=[paths[i] for i in test_idx],
    )


_IMAGE_CACHE: dict[Path, Image.Image] = {}


def load_rgb(path: Path, max_side: int = 768) -> Image.Image:
    cached = _IMAGE_CACHE.get(path)
    if cached is not None:
        return cached
    image = Image.open(path).convert("RGB")
    width, height = image.size
    scale = max_side / max(width, height)
    if scale < 1:
        image = image.resize((max(1, int(width * scale)), max(1, int(height * scale))), Image.Resampling.BICUBIC)
    _IMAGE_CACHE[path] = image
    return image


def crop_region(image: Image.Image, region: str) -> Image.Image:
    """Two views of the same field, not two different parts of the saree.

    Opposite corners of one photo are often the border and the body, so they
    are not a positive pair. Both crops sit on the center of the cloth.
    The query crop is shifted 48px and mirrored, so the pixels do not line
    up with the gallery crop and a hue-only change is not enough to match.
    """
    image = T.Resize(360)(image)
    width, height = image.size
    if region == "gallery":
        left = (width - 224) // 2
        top = (height - 224) // 2
        mirror = False
    elif region == "query":
        left = min(width - 224, (width - 224) // 2 + 48)
        top = min(height - 224, (height - 224) // 2 + 48)
        mirror = True
    elif region == "center":
        left = (width - 224) // 2
        top = (height - 224) // 2
        mirror = False
    else:
        raise ValueError(f"unknown region {region}")
    crop = image.crop((left, top, left + 224, top + 224))
    if mirror:
        crop = crop.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    return crop


def recolored_region(image: Image.Image, degrees: float = 0.0, sat_scale: float = 1.0, region: str = "center") -> Image.Image:
    if abs(degrees) > 1e-6 or abs(sat_scale - 1.0) > 1e-6:
        image = chroma_rotate(image, degrees, sat_scale)
    return crop_region(image, region)


def tensor_from_region(image: Image.Image) -> torch.Tensor:
    """image is already a 224 crop."""
    return _NORMALIZE(T.ToTensor()(image))


_CROP = T.RandomResizedCrop(224, scale=(0.55, 1.0), interpolation=T.InterpolationMode.BICUBIC)
_BLUR = T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
_NORMALIZE = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)


def _spatial_view(image: Image.Image) -> Image.Image:
    view = _CROP(image)
    if random.random() < 0.5:
        view = T.functional.hflip(view)
    if random.random() < 0.2:
        view = _BLUR(view)
    return view


def sample_batch(
    paths: list[Path],
    indices: np.ndarray,
    views: int,
    color_aug: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """P designs, K views each.

    View 0 of every design shares one hue, so the batch contains
    same-palette negatives. The other views use independent hues, so
    positives of one design do not share a palette.
    """
    shared_hue = random.uniform(0.0, 360.0)
    images: list[torch.Tensor] = []
    labels: list[int] = []
    for label, index in enumerate(indices.tolist()):
        image = load_rgb(paths[int(index)])
        for view_index in range(views):
            view = _spatial_view(image)
            if color_aug:
                if view_index == 0:
                    degrees, sat_scale = shared_hue, random.uniform(0.85, 1.15)
                elif random.random() < 0.15:
                    degrees, sat_scale = random.uniform(-20.0, 20.0), random.uniform(0.85, 1.15)
                else:
                    degrees = random.uniform(0.0, 360.0)
                    sat_scale = random.uniform(0.6, 1.4)
                view = chroma_rotate(view, degrees, sat_scale)
            images.append(_NORMALIZE(T.ToTensor()(view)))
            labels.append(label)
    return torch.stack(images), torch.tensor(labels, dtype=torch.long)


def iter_epoch(paths: list[Path], designs_per_batch: int, views: int, color_aug: bool, rng: np.random.Generator):
    order = rng.permutation(len(paths))
    for start in range(0, len(order) - designs_per_batch + 1, designs_per_batch):
        yield sample_batch(paths, order[start : start + designs_per_batch], views, color_aug)
''')
print('wrote', 'src/data.py', 'chars', len(Path('src/data.py').read_text()))


wrote src/data.py chars 5839


### `src/download.py`

In [8]:
from pathlib import Path
Path('src/download.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/download.py').write_text(r'''"""Download the DeepLure Drive corpus into data/.

The corpus is proprietary. It is gitignored and must not be redistributed.
Delete data/ once the exercise is over.
"""

from __future__ import annotations

import argparse
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

ROOT_FOLDER = "1V_DcXJ50QYV9nEqvq7fqKno_lhbSPdBM"
USER_AGENT = "Mozilla/5.0"


def _get(url: str) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        return response.read()


def list_folder(folder_id: str) -> list[tuple[str, str, bool]]:
    """Return (name, id, is_folder) for a publicly shared Drive folder."""
    html = _get(f"https://drive.google.com/embeddedfolderview?id={folder_id}").decode("utf-8", "replace")
    entries = re.findall(
        r'id="entry-([^"]+)"[\s\S]*?class="flip-entry-title"[^>]*>([^<]+)',
        html,
    )
    items = []
    for file_id, name in entries:
        entry_html = html.split(f'id="entry-{file_id}"', 1)[1].split('class="flip-entry"', 1)[0]
        is_folder = f"/folders/{file_id}" in entry_html
        items.append((name.strip(), file_id, is_folder))
    return items


def _download_file(file_id: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return
    data = _get(f"https://drive.google.com/uc?export=download&id={file_id}")
    if not data.startswith(b"\xff\xd8") and not data.startswith(b"\x89PNG"):
        match = re.search(br"confirm=([0-9A-Za-z_\-]+)", data)
        token = match.group(1).decode() if match else "t"
        data = _get(
            "https://drive.usercontent.google.com/download"
            f"?id={file_id}&export=download&confirm={token}"
        )
    if not data.startswith(b"\xff\xd8") and not data.startswith(b"\x89PNG"):
        raise RuntimeError(f"Drive did not return an image for {file_id} ({dest.name})")
    dest.write_bytes(data)


def download_corpus(dest_root: Path, workers: int = 6) -> int:
    dest_root.mkdir(parents=True, exist_ok=True)
    top = list_folder(ROOT_FOLDER)
    jobs: list[tuple[str, Path]] = []
    for name, folder_id, is_folder in top:
        if not is_folder:
            jobs.append((folder_id, dest_root / name))
            continue
        children = list_folder(folder_id)
        print(f"{name}: {len(children)} entries")
        for child_name, child_id, child_is_folder in children:
            if child_is_folder:
                continue
            jobs.append((child_id, dest_root / name / child_name))
    print(f"downloading {len(jobs)} files into {dest_root}")
    failed = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_download_file, fid, path): path for fid, path in jobs}
        done = 0
        for future in as_completed(futures):
            done += 1
            path = futures[future]
            try:
                future.result()
            except Exception as exc:
                failed += 1
                print(f"FAIL {path.name}: {exc}")
            if done % 25 == 0 or done == len(jobs):
                print(f"{done}/{len(jobs)}")
    if failed:
        print(f"{failed} downloads failed")
    return len(jobs) - failed


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dest", type=Path, default=Path("data/sarees"))
    parser.add_argument("--workers", type=int, default=6)
    args = parser.parse_args()
    count = download_corpus(args.dest, args.workers)
    print(f"ready: {count} images")


if __name__ == "__main__":
    main()
''')
print('wrote', 'src/download.py', 'chars', len(Path('src/download.py').read_text()))


wrote src/download.py chars 3730


### `src/train.py`

In [9]:
from pathlib import Path
Path('src/train.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/train.py').write_text(r'''"""Train the color-invariant saree encoder.

Each batch holds P designs and K views of each. Views are random crops,
and with --color-aug (the default) most of them are Lab hue rotations of
the same photo. The loss pulls those views together and pushes other
designs apart.
"""

from __future__ import annotations

import argparse
import json
import random
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

from src.data import GALLERY_HUE, QUERY_HUE, iter_epoch, list_images, split_designs
from src.loss import supcon_loss
from src.metrics import identification
from src.model import SareeEncoder


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def device_of() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def embed_split(model: SareeEncoder, paths: list[Path], degrees: float, region: str, device: torch.device, batch_size: int = 32) -> torch.Tensor:
    from src.data import load_rgb, recolored_region, tensor_from_region

    model.eval()
    outputs = []
    batch: list[torch.Tensor] = []
    for path in paths:
        batch.append(tensor_from_region(recolored_region(load_rgb(path), degrees, region=region)))
        if len(batch) == batch_size:
            outputs.append(model(torch.stack(batch).to(device)).cpu())
            batch = []
    if batch:
        outputs.append(model(torch.stack(batch).to(device)).cpu())
    return torch.cat(outputs)


def cross_palette_rank1(model: SareeEncoder, paths: list[Path], device: torch.device) -> float:
    gallery = embed_split(model, paths, GALLERY_HUE, "gallery", device)
    query = embed_split(model, paths, QUERY_HUE, "query", device)
    return identification(query, gallery)["rank1"]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=Path("data/sarees"))
    parser.add_argument("--out", type=Path, default=Path("runs/color"))
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--designs-per-batch", type=int, default=16)
    parser.add_argument("--views", type=int, default=4)
    parser.add_argument("--embed-dim", type=int, default=128)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--temperature", type=float, default=0.07)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--color-aug", action=argparse.BooleanOptionalAction, default=True)
    args = parser.parse_args()

    seed_everything(args.seed)
    device = device_of()
    args.out.mkdir(parents=True, exist_ok=True)

    paths = list_images(args.data)
    split = split_designs(paths, seed=args.seed)
    print(f"device={device} designs={len(paths)} train={len(split.train)} val={len(split.val)} test={len(split.test)}")
    print(f"color_aug={args.color_aug} views={args.views} designs_per_batch={args.designs_per_batch}")

    rng = np.random.default_rng(args.seed)
    steps = len(split.train) // args.designs_per_batch
    if steps == 0:
        raise RuntimeError("Training batch is larger than the train split. Lower --designs-per-batch.")

    model = SareeEncoder(embed_dim=args.embed_dim, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    history = []
    best_rank1 = -1.0
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        batches = iter_epoch(split.train, args.designs_per_batch, args.views, args.color_aug, rng)
        for images, labels in tqdm(batches, total=steps, desc=f"epoch {epoch}", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = supcon_loss(model(images), labels, temperature=args.temperature)
            if not torch.isfinite(loss):
                print("non-finite loss, skipping step")
                optimizer.zero_grad(set_to_none=True)
                continue
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
        schedule.step()
        rank1 = cross_palette_rank1(model, split.val, device)
        row = {"epoch": epoch, "loss": float(np.mean(losses)), "val_cross_palette_rank1": rank1}
        history.append(row)
        print(f"epoch {epoch:02d}  loss {row['loss']:.4f}  val rank-1 {rank1:.3f}")
        torch.save({"model": model.state_dict(), "args": vars(args) | {"out": str(args.out), "data": str(args.data)}, "epoch": epoch}, args.out / "last.pt")
        if rank1 >= best_rank1:
            best_rank1 = rank1
            torch.save({"model": model.state_dict(), "args": vars(args) | {"out": str(args.out), "data": str(args.data)}, "epoch": epoch, "val_rank1": rank1}, args.out / "best.pt")

    (args.out / "history.json").write_text(json.dumps(history, indent=2))
    print(f"best val rank-1 {best_rank1:.3f}  saved {args.out / 'best.pt'}")


if __name__ == "__main__":
    main()
''')
print('wrote', 'src/train.py', 'chars', len(Path('src/train.py').read_text()))


wrote src/train.py chars 5361


### `src/eval.py`

In [10]:
from pathlib import Path
Path('src/eval.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/eval.py').write_text(r'''"""Evaluation protocol.

The release has no real multi-colorway pairs, so colorways are synthetic
and the protocol says so.

Split: by source photo, seed 42, 60% train / 20% val / 20% test.
Train photos are never embedded in the reported numbers.

Cross-palette identification
  Gallery: center 224 crop, Lab hue +40°.
  Query: the center crop shifted 48px, mirrored, Lab hue +160°.
  Both crops are the field of the cloth, not the border versus the body.
  The shift and the mirror keep the pixels from lining up.
  Metrics: Rank-1, Rank-5, mAP. One relevant gallery item per query,
  so AP is 1 / rank of that item.

Verification
  Positive pairs: query_i with gallery_i.
  Negative pairs: query_i with gallery_j, j != i.
  Metrics: ROC-AUC, equal-error rate, TAR at FAR = 1%.
  The decision threshold is chosen on val (EER) and applied to test.

Same-palette stress test
  Every gallery photo is recolored so its dominant hue matches that
  query. Rank-1 here is the claim that the motif, not the palette, is
  what matched.

An unedited-photo check embeds the shifted, mirrored crop with its
original palette against the recolored gallery crop.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch

from src.color import palette_match
from src.data import (
    GALLERY_HUE,
    QUERY_HUE,
    list_images,
    load_rgb,
    recolored_region,
    split_designs,
    tensor_from_region,
)
from src.metrics import identification, pair_scores, verification
from src.model import SareeEncoder


def device_of() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def load_model(checkpoint: Path, device: torch.device) -> tuple[SareeEncoder, dict]:
    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
    args = saved["args"]
    model = SareeEncoder(embed_dim=args["embed_dim"], pretrained=False)
    model.load_state_dict(saved["model"])
    model.to(device).eval()
    return model, saved


@torch.no_grad()
def embed_images(
    model: SareeEncoder,
    images,
    device: torch.device,
    batch_size: int = 32,
    backbone_only: bool = False,
) -> torch.Tensor:
    encode = model.forward_backbone if backbone_only else model
    outputs = []
    batch = []
    for image in images:
        batch.append(tensor_from_region(image))
        if len(batch) == batch_size:
            outputs.append(encode(torch.stack(batch).to(device)).cpu())
            batch = []
    if batch:
        outputs.append(encode(torch.stack(batch).to(device)).cpu())
    return torch.cat(outputs)


def region_views(paths: list[Path], degrees: float, region: str):
    return [recolored_region(load_rgb(path), degrees, region=region) for path in paths]


def report_split(
    name: str,
    model,
    paths: list[Path],
    device,
    threshold: float | None = None,
    backbone_only: bool = False,
) -> dict:
    gallery_images = region_views(paths, GALLERY_HUE, "gallery")
    query_images = region_views(paths, QUERY_HUE, "query")
    originals = region_views(paths, 0.0, "query")
    gallery = embed_images(model, gallery_images, device, backbone_only=backbone_only)
    query = embed_images(model, query_images, device, backbone_only=backbone_only)
    original = embed_images(model, originals, device, backbone_only=backbone_only)

    cross = identification(query, gallery)
    original_as_query = identification(original, gallery)
    positive, negative = pair_scores(query, gallery)
    pairs = verification(positive, negative)
    if threshold is not None:
        pairs["tar_at_val_threshold"] = float((positive >= threshold).mean())
        pairs["far_at_val_threshold"] = float((negative >= threshold).mean())
        pairs["val_threshold"] = threshold

    # Same-palette impostors: recolor every gallery item toward this query.
    stress_ranks = []
    for index, query_image in enumerate(query_images):
        matched = [palette_match(gallery_image, query_image) for gallery_image in gallery_images]
        matched_emb = embed_images(model, matched, device, backbone_only=backbone_only)
        query_emb = embed_images(model, [query_image], device, backbone_only=backbone_only)
        similarity = (query_emb @ matched_emb.T).squeeze(0)
        order = torch.argsort(similarity, descending=True)
        rank = int(order.eq(index).nonzero(as_tuple=False)[0].item()) + 1
        stress_ranks.append(rank)
    stress = torch.tensor(stress_ranks)
    return {
        "split": name,
        "designs": len(paths),
        "cross_palette_identification": cross,
        "original_query_identification": original_as_query,
        "cross_palette_verification": pairs,
        "same_palette_stress": {
            "rank1": float((stress == 1).float().mean()),
            "rank5": float((stress <= 5).float().mean()),
            "mAP": float((1.0 / stress.float()).mean()),
        },
    }


def load_imagenet_baseline(device: torch.device) -> SareeEncoder:
    """ImageNet MobileNetV3-Small. Embeddings are pooled features, not the projection."""
    model = SareeEncoder(pretrained=True)
    model.to(device).eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    return model


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, default=Path("runs/color/best.pt"))
    parser.add_argument("--data", type=Path, default=None)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", type=Path, default=None)
    parser.add_argument(
        "--imagenet",
        action="store_true",
        help="Frozen ImageNet backbone. L2-normalized pooled features, no trained projection.",
    )
    args = parser.parse_args()

    device = device_of()
    if args.imagenet:
        model = load_imagenet_baseline(device)
        saved = {"epoch": None, "args": {"data": "data/sarees", "embed_dim": model.feat_dim}}
        data_root = Path(args.data or "data/sarees")
        out = args.out or Path("runs/imagenet/eval.json")
        representation = "ImageNet MobileNetV3-Small pooled features, L2-normalized, no fine-tuning and no projection"
    else:
        model, saved = load_model(args.checkpoint, device)
        data_root = Path(args.data or saved["args"]["data"])
        out = args.out or (args.checkpoint.parent / "eval.json")
        representation = f"fine-tuned projection, {model.embed_dim}-d"
    split = split_designs(list_images(data_root), seed=args.seed)

    # Threshold is selected on val. Test numbers below use the val EER threshold
    # only as a reported operating point; AUC and EER are threshold-free.
    val_report = report_split("val", model, split.val, device, backbone_only=args.imagenet)
    val_threshold = val_report["cross_palette_verification"]["eer_threshold"]
    test_report = report_split("test", model, split.test, device, threshold=val_threshold, backbone_only=args.imagenet)
    payload = {
        "checkpoint": "imagenet" if args.imagenet else str(args.checkpoint),
        "epoch": saved.get("epoch"),
        "representation": representation,
        "embedding_dim": model.feat_dim if args.imagenet else model.embed_dim,
        "protocol": {
            "unit": "one source photo = one design",
            "colorways": "synthetic Lab chroma rotation; the corpus has no colorway labels",
            "gallery_hue_deg": GALLERY_HUE,
            "query_hue_deg": QUERY_HUE,
            "gallery_crop": "center 224 of a 360px resize, hue +40",
            "query_crop": "center 224 shifted 48px and mirrored, hue +160",
            "split": "60/20/20 by photo, seed 42",
            "corpus": "Drive handloom_sarees only (165 photos). normal_sarees was empty in the public folder. Kaggle weave-class set is not design-level and was not used.",
            "verification_threshold": "EER threshold fit on val",
            "val_eer_threshold": val_report["cross_palette_verification"]["eer_threshold"],
            "baseline": args.imagenet,
        },
        "val": val_report,
        "test": test_report,
    }
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(payload, indent=2))
    print(json.dumps({"val": val_report, "test": test_report}, indent=2))
    print(f"wrote {out}")


if __name__ == "__main__":
    main()
''')
print('wrote', 'src/eval.py', 'chars', len(Path('src/eval.py').read_text()))


wrote src/eval.py chars 6791


### `src/efficiency.py`

In [11]:
from pathlib import Path
Path('src/efficiency.py').parent.mkdir(parents=True, exist_ok=True)
Path('src/efficiency.py').write_text(r'''"""Parameter count, conv/linear FLOPs, and single-image latency."""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import torch
from torch import nn

from src.model import SareeEncoder


def device_of() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def count_conv_linear_flops(model: nn.Module, device: torch.device) -> int:
    total = 0

    def conv_hook(module, inputs, output):
        nonlocal total
        n, _, h, w = output.shape
        kh, kw = module.kernel_size
        total += int(2 * n * module.in_channels * kh * kw * h * w * module.out_channels / module.groups)

    def linear_hook(module, inputs, output):
        nonlocal total
        batch = output.shape[0]
        total += int(2 * batch * module.in_features * module.out_features)

    handles = []
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            handles.append(module.register_forward_hook(conv_hook))
        elif isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(linear_hook))
    with torch.no_grad():
        model(torch.zeros(1, 3, 224, 224, device=device))
    for handle in handles:
        handle.remove()
    return total


def latency_ms(model: nn.Module, device: torch.device, iters: int = 100) -> float:
    sample = torch.zeros(1, 3, 224, 224, device=device)
    sync = torch.cuda.synchronize if device.type == "cuda" else (torch.mps.synchronize if device.type == "mps" else lambda: None)
    with torch.no_grad():
        for _ in range(20):
            model(sample)
        sync()
        start = time.perf_counter()
        for _ in range(iters):
            model(sample)
        sync()
    return (time.perf_counter() - start) * 1000.0 / iters


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, default=Path("runs/color/best.pt"))
    args = parser.parse_args()

    device = device_of()
    saved = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    model = SareeEncoder(embed_dim=saved["args"]["embed_dim"], pretrained=False)
    model.load_state_dict(saved["model"])
    model.to(device).eval()

    params = sum(p.numel() for p in model.parameters())
    flops = count_conv_linear_flops(model, device)
    report = {
        "backbone": "torchvision MobileNetV3-Small, ImageNet-1K weights, then fine-tuned",
        "parameters": params,
        "parameters_millions": round(params / 1e6, 3),
        "conv_linear_flops": flops,
        "conv_linear_gflops": round(flops / 1e9, 3),
        "embedding_dim": model.embed_dim,
        "input": "1x3x224x224",
        "latency_ms_batch1": round(latency_ms(model, device), 2),
        "device": str(device),
        "notes": "FLOPs count multiply-adds in Conv2d and Linear only.",
    }
    out = args.checkpoint.parent / "efficiency.json"
    out.write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
    print(f"wrote {out}")


if __name__ == "__main__":
    main()
''')
print('wrote', 'src/efficiency.py', 'chars', len(Path('src/efficiency.py').read_text()))


wrote src/efficiency.py chars 3226


In [ ]:
from pathlib import Path
import shutil


input_root = Path("/kaggle/input")


images = [
    p for p in input_root.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]


print("Images found in /kaggle/input:", len(images))


if len(images) == 0:
    raise RuntimeError(
        "No saree images found in /kaggle/input. "
        "Check that the private handloom_saree dataset is attached."
    )


data = Path("data/sarees")
data.mkdir(parents=True, exist_ok=True)


for src in images:
    dst = data / src.name
    shutil.copy2(src, dst)


print("Images copied to:", data)
print("Images in data/sarees:", len(list(data.glob("*"))))


from src.data import list_images
print("Designs:", len(list_images(data)))


assert len(list_images(data)) >= 100, "Expected the DeepLure saree corpus to be present."


## Train

40 epochs, 16 designs per batch, 4 views. One view in the batch shares a hue across designs, so different motifs in the same palette are negatives. The other views are independent recolorings of the same photo.


In [13]:
import sys
sys.argv = [
    "train",
    "--data", "data/sarees",
    "--out", "runs/color",
    "--epochs", "40",
    "--designs-per-batch", "16",
    "--views", "4",
]
from src.train import main
main()


device=mps designs=165 train=99 val=33 test=33
color_aug=True views=4 designs_per_batch=16


epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

epoch 1:  17%|█▋        | 1/6 [00:01<00:09,  1.98s/it]

epoch 1:  33%|███▎      | 2/6 [00:02<00:04,  1.22s/it]

epoch 1:  50%|█████     | 3/6 [00:03<00:02,  1.05it/s]

...

epoch 40:  50%|█████     | 3/6 [00:01<00:01,  1.78it/s]

epoch 40:  67%|██████▋   | 4/6 [00:02<00:01,  1.82it/s]

epoch 40:  83%|████████▎ | 5/6 [00:02<00:00,  1.84it/s]

epoch 40: 100%|██████████| 6/6 [00:03<00:00,  1.85it/s]

epoch 40  loss 1.2376  val rank-1 0.970
best val rank-1 0.970  saved runs/color/best.pt


## Evaluation

In [14]:
import sys
sys.argv = ["eval", "--checkpoint", "runs/color/best.pt"]
from src.eval import main
main()


{
  "val": {
    "split": "val",
    "designs": 33,
    "cross_palette_identification": {
      "rank1": 0.9696969985961914,
      "rank5": 1.0,
      "mAP": 0.9848484992980957
    },
    "original_query_identification": {
      "rank1": 0.9696969985961914,
      "rank5": 1.0,
      "mAP": 0.9848484992980957
    },
    "cross_palette_verification": {
      "roc_auc": 0.9962695133149678,
      "eer": 0.025568181818181816,
      "eer_threshold": 0.665,
      "tar_at_far_1pct": 0.9393939393939394
    },
    "same_palette_stress": {
      "rank1": 0.9696969985961914,
      "rank5": 1.0,
      "mAP": 0.9848484992980957
    }
  },
  "test": {
    "split": "test",
    "designs": 33,
    "cross_palette_identification": {
      "rank1": 0.8787878751754761,
      "rank5": 1.0,
      "mAP": 0.939393937587738
    },
    "original_query_identification": {
      "rank1": 0.8787878751754761,
      "rank5": 1.0,
      "mAP": 0.939393937587738
    },
    "cross_palette_verification": {
      "roc_auc":

## Efficiency

In [15]:
import sys
sys.argv = ["efficiency", "--checkpoint", "runs/color/best.pt"]
from src.efficiency import main
main()


{
  "backbone": "torchvision MobileNetV3-Small, ImageNet-1K weights, then fine-tuned",
  "parameters": 1000992,
  "parameters_millions": 1.001,
  "conv_linear_flops": 109940608,
  "conv_linear_gflops": 0.11,
  "embedding_dim": 128,
  "input": "1x3x224x224",
  "latency_ms_batch1": 4.96,
  "device": "mps",
  "notes": "FLOPs count multiply-adds in Conv2d and Linear only."
}
wrote runs/color/efficiency.json


## Reported numbers

In [16]:
import json
from pathlib import Path

eval_path = Path("runs/color/eval.json")
eff_path = Path("runs/color/efficiency.json")
if not eval_path.exists():
    raise FileNotFoundError(
        f"{eval_path} not found. Run the cells in order: Train (cell 24) "
        "creates runs/color/best.pt, then Evaluation (cell 26) writes this file. "
        "Do not run this cell standalone after a kernel restart."
    )
if not eff_path.exists():
    raise FileNotFoundError(
        f"{eff_path} not found. Run the Efficiency cell (cell 28) first: "
        "it measures the trained model and writes this file."
    )
report = json.loads(eval_path.read_text())
eff = json.loads(eff_path.read_text())
test = report["test"]
ident = test["cross_palette_identification"]
ver = test["cross_palette_verification"]
stress = test["same_palette_stress"]
original = test["original_query_identification"]
print(f"test designs                  {test['designs']}")
print(f"cross-palette Rank-1          {ident['rank1']*100:.1f}%")
print(f"cross-palette Rank-5          {ident['rank5']*100:.1f}%")
print(f"cross-palette mAP             {ident['mAP']*100:.1f}%")
print(f"original-photo Rank-1         {original['rank1']*100:.1f}%")
print(f"verification ROC-AUC          {ver['roc_auc']:.3f}")
print(f"verification EER              {ver['eer']*100:.1f}%")
print(f"TAR at FAR 1%                 {ver['tar_at_far_1pct']*100:.1f}%")
print(f"TAR at val threshold          {ver['tar_at_val_threshold']*100:.1f}%")
print(f"FAR at val threshold          {ver['far_at_val_threshold']*100:.1f}%")
print(f"same-palette stress Rank-1    {stress['rank1']*100:.1f}%")
print(f"parameters                    {eff['parameters_millions']} M")
print(f"conv/linear FLOPs             {eff['conv_linear_gflops']} GFLOPs")
print(f"embedding                     {eff['embedding_dim']}")
print(f"latency                       {eff['latency_ms_batch1']} ms on {eff['device']}")


test designs                  33
cross-palette Rank-1          87.9%
cross-palette Rank-5          100.0%
cross-palette mAP             93.9%
original-photo Rank-1         87.9%
verification ROC-AUC          0.992
verification EER              3.1%
TAR at FAR 1%                 75.8%
TAR at val threshold          97.0%
FAR at val threshold          4.4%
same-palette stress Rank-1    90.9%
parameters                    1.001 M
conv/linear FLOPs             0.11 GFLOPs
embedding                     128
latency                       4.96 ms on mps


## Retrieval strips

Query crop at hue +160° (shifted, mirrored), then its top-5 gallery crops at hue +40° with cosine scores. The true match is the same photo in a disjoint palette; on the held-out test set it ranks top-1 for 30 of 33 queries.

In [ ]:
import torch
from pathlib import Path
from IPython.display import display
from PIL import Image

from src.data import GALLERY_HUE, QUERY_HUE, list_images, split_designs
from src.eval import device_of, embed_images, load_model, region_views

ckpt32 = Path("runs/color/best.pt")
if not ckpt32.exists():
    raise FileNotFoundError(
        "runs/color/best.pt not found. Run the Train cell (cell 24) first: "
        "it trains 40 epochs and saves the checkpoint."
    )
strip_device = device_of()
strip_model, _ = load_model(Path("runs/color/best.pt"), strip_device)
strip_split = split_designs(list_images(Path("data/sarees")), seed=42)
strip_gallery_images = region_views(strip_split.test, GALLERY_HUE, "gallery")
strip_query_images = region_views(strip_split.test, QUERY_HUE, "query")
strip_gallery = embed_images(strip_model, strip_gallery_images, strip_device)
strip_query = embed_images(strip_model, strip_query_images, strip_device)
strip_sim = strip_query @ strip_gallery.T

for strip_qi in [0, 1, 2]:
    strip_order = strip_sim[strip_qi].argsort(descending=True)[:5]
    tiles = [strip_query_images[strip_qi]] + [strip_gallery_images[i] for i in strip_order]
    scores = ["query"] + [f"{float(strip_sim[strip_qi][i]):.3f}" for i in strip_order]
    w, h = tiles[0].size
    canvas = Image.new("RGB", (w * len(tiles), h + 22), (255, 255, 255))
    canvas.paste(strip_query_images[strip_qi], (0, 22))
    for k, t in enumerate(tiles[1:], start=1):
        canvas.paste(t, (k * w, 22))
    print(f"query {strip_qi} ({strip_split.test[strip_qi].name}) top-5 scores: " + ", ".join(scores[1:]))
    display(canvas)


## Color-invariance demo

One held-out photo rendered at eight disjoint hues. Cosine similarity against the original palette stays high: the embedding tracks the motif, not the palette.

In [ ]:
import torch
from pathlib import Path

from src.color import chroma_rotate
from src.data import list_images, load_rgb, recolored_region, split_designs, tensor_from_region
from src.eval import device_of, load_model

ckpt34 = Path("runs/color/best.pt")
if not ckpt34.exists():
    raise FileNotFoundError(
        "runs/color/best.pt not found. Run the Train cell (cell 24) first."
    )
inv_device = device_of()
inv_model, _ = load_model(Path("runs/color/best.pt"), inv_device)
inv_photo = split_designs(list_images(Path("data/sarees")), seed=42).test[0]
inv_base = load_rgb(inv_photo)
inv_hues = [0, 40, 80, 120, 160, 200, 240, 280]
with torch.no_grad():
    inv_embs = torch.stack([
        inv_model(tensor_from_region(recolored_region(chroma_rotate(inv_base, h), region="center")).unsqueeze(0).to(inv_device)).cpu().squeeze(0)
        for h in inv_hues
    ])
inv_sims = (inv_embs @ inv_embs[0]).tolist()
for h, s in zip(inv_hues, inv_sims):
    print(f"hue +{h:3d}deg  cosine vs original palette: {s:.3f}")
print("Same photo under disjoint palettes stays highly similar: design, not color, is matched.")


## Verification detail

Accuracy, precision, recall, and F1 at the validation-selected threshold, plus score distributions and the test ROC curve. Precision is modest by construction: each query has 1 positive and 32 negatives.

In [ ]:
import numpy as np

try:
    report, strip_query, strip_gallery
except NameError as exc:
    raise NameError(
        "Missing variables from earlier cells. Run in order: Reported numbers "
        "(cell 30) defines 'report'; Retrieval strips (cell 32) defines "
        "'strip_query'/'strip_gallery'. Then re-run this cell."
    ) from exc

from src.metrics import pair_scores

ver_thr = report["protocol"]["val_eer_threshold"]
ver_pos, ver_neg = pair_scores(strip_query, strip_gallery)
ver_TP, ver_FN = int((ver_pos >= ver_thr).sum()), int((ver_pos < ver_thr).sum())
ver_FP, ver_TN = int((ver_neg >= ver_thr).sum()), int((ver_neg < ver_thr).sum())
ver_acc = (ver_TP + ver_TN) / (ver_TP + ver_TN + ver_FP + ver_FN)
ver_prec = ver_TP / (ver_TP + ver_FP)
ver_rec = ver_TP / (ver_TP + ver_FN)
ver_f1 = 2 * ver_prec * ver_rec / (ver_prec + ver_rec)
print(f"val threshold {ver_thr:.2f}  accuracy {ver_acc:.3f}  precision {ver_prec:.3f}  "
      f"recall {ver_rec:.3f}  F1 {ver_f1:.3f}")
print(f"TP {ver_TP} FN {ver_FN} FP {ver_FP} TN {ver_TN} "
      f"(positives {len(ver_pos)}, negatives {len(ver_neg)}: precision is low by construction)")
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(ver_neg, bins=40, alpha=0.6, label="different-design")
    ax[0].hist(ver_pos, bins=20, alpha=0.7, label="same-design")
    ax[0].axvline(ver_thr, color="k", ls="--", label=f"val thr {ver_thr:.2f}")
    ax[0].legend()
    ax[0].set_title("Verification score distributions (test)")
    ax[0].set_xlabel("cosine similarity")
    ts = np.linspace(-1, 1, 401)
    tpr = [(ver_pos >= t).mean() for t in ts]
    fpr = [(ver_neg >= t).mean() for t in ts]
    ax[1].plot(fpr, tpr)
    ax[1].plot([0, 1], [0, 1], "k--")
    ax[1].set_title("ROC (test)")
    ax[1].set_xlabel("FAR")
    ax[1].set_ylabel("TAR")
    plt.show()
except ImportError:
    print("matplotlib unavailable; skipping plots.")
